<!-- 說明：本筆記本用於在 Google Colab L4 GPU 環境下執行 Amidst Q1–Q5 Agent 開源權重模型（Qwen2.5-7B-Instruct / Llama-3.1-8B-Instruct, q4_K_M）決策評估。 -->
# Amidst Q1–Q5 Agent 評估（Colab L4 開源權重）

本筆記本依序執行：
1. 檢查 GPU 型號是否為 NVIDIA L4（若非 L4 則僅記錄決策，不宣稱正式延遲）。
2. 執行 `tools/verify_pack.py` 驗證實驗包 SHA-256 與無禁止欄位。
3. 啟動並檢查 Ollama `/api/tags` 與 `models.txt`，鎖定模型 digest 與 `q4_K_M` 量化。
4. 先執行 `DRY_RUN` 確認題數與預算，再執行推論並將產物寫入 `RETURN/`。

In [ ]:
# 步驟 1：檢查 Colab GPU 型號是否為 NVIDIA L4；若不是 L4，設定 measure_latency=False（僅記錄決策，不計正式延遲）。
import json
import subprocess
from amidst.agent_eval.runner import detect_colab_gpu

proc = subprocess.run(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    capture_output=True,
    text=True,
    check=False,
)
gpu_info = detect_colab_gpu(proc.stdout if proc.returncode == 0 else "")
print(json.dumps(gpu_info, ensure_ascii=False, indent=2))
MEASURE_LATENCY = gpu_info["measure_latency"]


In [ ]:
# 步驟 2：執行 verify_pack.py 驗證上傳之實驗包完整性、相對路徑與無 GT/私有標籤洩漏。
import subprocess
from pathlib import Path

PACK_DIR = Path(".")
verify_proc = subprocess.run(
    ["python3", str(PACK_DIR / "tools" / "verify_pack.py"), str(PACK_DIR)],
    capture_output=True,
    text=True,
    check=False,
)
print(verify_proc.stdout.strip())
assert verify_proc.returncode == 0 and "VALID" in verify_proc.stdout, verify_proc.stdout + verify_proc.stderr


In [ ]:
# 步驟 3：核對 Ollama /api/tags 與 models.txt，鎖定 Qwen2.5-7B-Instruct 與 Llama-3.1-8B-Instruct 的 digest 及 q4_K_M 量化。
import json
import urllib.request
from pathlib import Path
from amidst.agent_eval.runner import verify_ollama_models

models_txt_path = Path("models.txt")
models_txt = models_txt_path.read_text(encoding="utf-8") if models_txt_path.is_file() else (
    "qwen2.5:7b-instruct-q4_K_M\nllama3.1:8b-instruct-q4_K_M\n"
)
with urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=5) as resp:
    tags_payload = json.loads(resp.read().decode("utf-8"))

model_check = verify_ollama_models(models_txt, tags_payload)
print(json.dumps(model_check, ensure_ascii=False, indent=2))
assert model_check["valid"], f"Ollama model check failed: {model_check}"


In [ ]:
# 步驟 4：先執行 DRY_RUN 確認題數與預算（0 次呼叫），再將產物與環境摘要寫入 RETURN/ 供回傳。
from pathlib import Path
from amidst.agent_eval import runner

rc = runner.main([
    "--questions", "data/requests_v1.jsonl",
    "--params", "configs/q1q5/agent_decision_params_v1.json",
    "--stage", "PILOT_AND_CONSISTENCY",
    "--k", "3",
    "--out", "RETURN/dry_run_check",
])
assert rc == 0, f"DRY_RUN failed with rc={rc}"
print(Path("RETURN/dry_run_check/dry_run.json").read_text(encoding="utf-8"))
